In [1]:
print("Installing Required Packages...")
!pip install -q torch torchvision torchaudio transformers opencv-python numpy timm einops
print("Required packages have already been installed!")
import os
import torch
import torchvision.transforms as transforms
from transformers import AutoImageProcessor, AutoModelForVideoClassification
import cv2
import numpy as np
import time
from torch.nn import functional as F
import json

# ===== Mount Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

# ===== CONFIG =====
VIDEO_PATH = '/content/drive/MyDrive/AnamolyDetection/Abuse/Abuse001_x264.mp4'  # Your test video
MODEL_PATH = '/content/drive/MyDrive/crime_classification_checkpoints/best_model.pth'  # Fine-tuned model path
PROCESSOR_PATH = '/content/drive/MyDrive/pretrained_models/TimeSformer_divST_8x32_224_K400.pyth'  # Path to saved processor
CONFIG_PATH = '/content/drive/MyDrive/timesformer_config.json'  # Path to saved model config (optional)
NUM_FRAMES = 16
NUM_CLASSES = 14
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print("[INFO] Environment setup done.")
print(f"[INFO] Using device: {DEVICE}")

# ===== Load Config (if available) =====
config = None
if os.path.exists(CONFIG_PATH):
    try:
        print(f"[INFO] Loading model config from {CONFIG_PATH}...")
        with open(CONFIG_PATH, 'r') as f:
            config = json.load(f)
        print("[INFO] Successfully loaded model config.")
    except Exception as e:
        print(f"[WARNING] Could not load config: {e}")
        config = None

# ===== Load Model =====
print(f"[INFO] Loading model weights from {MODEL_PATH}...")
try:
    # Check what's in the file - is it a full model or just state_dict?
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

    # If checkpoint is a state_dict (OrderedDict)
    if isinstance(checkpoint, dict) and not isinstance(checkpoint, torch.nn.Module):
        print("[INFO] Loaded file contains a state dictionary, not a full model.")
        state_dict = checkpoint

        # Initialize the base model
        print("[INFO] Initializing base TimeSformer model...")
        model = AutoModelForVideoClassification.from_pretrained("facebook/timesformer-base-finetuned-k400")

        # Check if we need to adjust the classifier
        if model.classifier.out_features != NUM_CLASSES:
            print(f"[INFO] Replacing classifier head for {NUM_CLASSES} classes...")
            model.classifier = torch.nn.Linear(model.classifier.in_features, NUM_CLASSES)

        # Load state dict with flexible settings to handle partial matches
        try:
            # Try strict loading first
            model.load_state_dict(state_dict)
            print("[INFO] Successfully loaded weights with strict matching.")
        except Exception as e:
            print(f"[WARNING] Strict loading failed: {e}")
            print("[INFO] Trying non-strict loading...")
            model.load_state_dict(state_dict, strict=False)
            print("[INFO] Successfully loaded weights with non-strict matching.")
    else:
        # If checkpoint is a full model
        print("[INFO] Loaded file contains a full model.")
        model = checkpoint

except Exception as e:
    print(f"[ERROR] Failed to load model: {e}")
    # Last resort - initialize fresh model
    print("[INFO] Initializing fresh TimeSformer model...")
    model = AutoModelForVideoClassification.from_pretrained("facebook/timesformer-base-finetuned-k400")
    if model.classifier.out_features != NUM_CLASSES:
        model.classifier = torch.nn.Linear(model.classifier.in_features, NUM_CLASSES)
    print("[WARNING] Using fresh model with default weights!")

# Move model to device and set to evaluation mode
model.to(DEVICE)
model.eval()
print("[INFO] Model loaded successfully.")
print(f"[DEBUG] Model classifier out_features: {model.classifier.out_features}")

# ===== Define Class Names =====
# Replace with your crime classes
class_names = [
    "Abuse", "Arrest", "Arson", "Assault", "Burglary",
    "Explosion", "Fighting", "Robbery", "Shooting", "Shoplifting",
    "Stealing", "Vandalism", "Vehicle Theft", "Weapon Display"
]

# ===== Frame Extraction =====
def extract_frames(video_path, num_frames=16):
    print(f"[INFO] Extracting {num_frames} frames from: {video_path}")
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"[DEBUG] Total frames in video: {total_frames}")

    if total_frames < num_frames:
        raise ValueError("Not enough frames in video.")

    indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
    frames = []
    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        if i in indices:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(rgb_frame)
    cap.release()

    print(f"[INFO] Extracted {len(frames)} frames.")
    return frames

# ===== Load Image Processor =====
print("[INFO] Loading image processor...")
# Always use the default processor instead of trying to load from file
# This avoids the TypeError when trying to call a dict object
processor = AutoImageProcessor.from_pretrained("facebook/timesformer-base-finetuned-k400")
print("[INFO] Successfully loaded default TimeSformer processor.")

# ===== Define preprocessing function =====
def preprocess_frames(frames):
    print("[INFO] Preprocessing frames...")
    # Create a simple preprocessing pipeline if processor is a dict
    if isinstance(frames[0], np.ndarray):
        # Convert numpy frames to torch tensors and normalize
        processed_frames = []
        for frame in frames:
            # Convert to torch tensor and normalize
            transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Resize((224, 224)),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            processed_frames.append(transform(frame))

        # Stack frames
        video_tensor = torch.stack(processed_frames)
        # Add batch dimension
        video_tensor = video_tensor.unsqueeze(0)
        inputs = {'pixel_values': video_tensor}
    else:
        # Use the processor directly if it's callable
        inputs = processor(videos=[frames], return_tensors="pt")

    print(f"[DEBUG] Input shape: {inputs['pixel_values'].shape}")
    return inputs

# ===== Inference with Metrics =====
frames = extract_frames(VIDEO_PATH, NUM_FRAMES)
inputs = preprocess_frames(frames)
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

print("[INFO] Running inference...")
# Measure inference time
start_time = time.time()
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    # Apply softmax to get probabilities
    probabilities = F.softmax(logits, dim=1)

    # Get top prediction and confidence score
    confidence, predicted_index = torch.max(probabilities, dim=1)

    # Get top-3 predictions and scores
    top3_confidence, top3_indices = torch.topk(probabilities, k=min(3, NUM_CLASSES), dim=1)

inference_time = time.time() - start_time

# Extract results
predicted_class_idx = predicted_index.item()
confidence_score = confidence.item()

# Print results
print("\n===== INFERENCE RESULTS =====")
print(f"Inference Time: {inference_time:.4f} seconds")
print(f"Predicted Class: {predicted_class_idx} - {class_names[predicted_class_idx] if len(class_names) == NUM_CLASSES else 'Unknown'}")
print(f"Confidence Score: {confidence_score:.4f} ({confidence_score*100:.2f}%)")

# Print top-3 predictions
print("\nTop 3 Predictions:")
for i in range(min(3, NUM_CLASSES)):
    idx = top3_indices[0, i].item()
    score = top3_confidence[0, i].item()
    print(f"  {i+1}. {class_names[idx] if len(class_names) == NUM_CLASSES else f'Class {idx}'}: {score:.4f} ({score*100:.2f}%)")

Installing Required Packages...
Required packages have already been installed!
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Environment setup done.
[INFO] Using device: cuda
[INFO] Loading model weights from /content/drive/MyDrive/crime_classification_checkpoints/best_model.pth...
[INFO] Loaded file contains a state dictionary, not a full model.
[INFO] Initializing base TimeSformer model...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[INFO] Replacing classifier head for 14 classes...
[WARNING] Strict loading failed: Error(s) in loading state_dict for TimesformerForVideoClassification:
	Missing key(s) in state_dict: "timesformer.embeddings.cls_token", "timesformer.embeddings.position_embeddings", "timesformer.embeddings.time_embeddings", "timesformer.embeddings.patch_embeddings.projection.weight", "timesformer.embeddings.patch_embeddings.projection.bias", "timesformer.encoder.layer.0.attention.attention.qkv.weight", "timesformer.encoder.layer.0.attention.attention.qkv.bias", "timesformer.encoder.layer.0.attention.output.dense.weight", "timesformer.encoder.layer.0.attention.output.dense.bias", "timesformer.encoder.layer.0.intermediate.dense.weight", "timesformer.encoder.layer.0.intermediate.dense.bias", "timesformer.encoder.layer.0.output.dense.weight", "timesformer.encoder.layer.0.output.dense.bias", "timesformer.encoder.layer.0.layernorm_before.weight", "timesformer.encoder.layer.0.layernorm_before.bias", "timesfor

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[INFO] Successfully loaded default TimeSformer processor.
[INFO] Extracting 16 frames from: /content/drive/MyDrive/AnamolyDetection/Abuse/Abuse001_x264.mp4
[DEBUG] Total frames in video: 2729
[INFO] Extracted 16 frames.
[INFO] Preprocessing frames...
[DEBUG] Input shape: torch.Size([1, 16, 3, 224, 224])
[INFO] Running inference...

===== INFERENCE RESULTS =====
Inference Time: 1.2896 seconds
Predicted Class: 4 - Burglary
Confidence Score: 0.1690 (16.90%)

Top 3 Predictions:
  1. Burglary: 0.1690 (16.90%)
  2. Arrest: 0.1183 (11.83%)
  3. Fighting: 0.0996 (9.96%)
